In [ ]:
%%capture install_log
!pip install -q ultralytics pydicom opencv-python-headless tqdm pandas openpyxl scikit-learn pyyaml

In [ ]:
!pip install monai

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import pydicom
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import monai
from monai.transforms import (
    Compose, 
    EnsureChannelFirstd, 
    ScaleIntensityRanged, 
    Resized, 
    ToTensord,
    RandRotate90d,
    RandFlipd,
    RandGaussianNoised,
    RandZoomd
)
from sklearn.model_selection import GroupKFold
from tqdm import tqdm


In [ ]:
# Cihaz Kontrolü
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Aktif Cihaz: {device}")

# T4 x2: kaç GPU var?
n_gpu = torch.cuda.device_count()
print(f"GPU sayısı: {n_gpu}")

# Kaggle Dizin Yolları
DATA_DIR = "/kaggle/input/competitions/rsna-2023-abdominal-trauma-detection"
TRAIN_IMAGES_DIR = os.path.join(DATA_DIR, "train_images")
TRAIN_CSV = os.path.join(DATA_DIR, "train_2024.csv")


In [ ]:
# DICOM -> NPY Onbellek
# Bir kez calistir, sonra hep .npy oku.
# 128x128x128 float16 -> hasta basina ~4 MB, 3147 hasta -> ~12 GB
import os, glob
import numpy as np
import pydicom
from scipy.ndimage import zoom
from tqdm import tqdm

CACHE_DIR = '/kaggle/working/npy_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

def dicom_to_npy(patient_id: int):
    patient_dir = os.path.join(TRAIN_IMAGES_DIR, str(patient_id))
    series_ids = sorted(os.listdir(patient_dir))
    if not series_ids:
        return None
    series_dir = os.path.join(patient_dir, series_ids[0])
    dcm_files  = glob.glob(os.path.join(series_dir, '*.dcm'))
    if not dcm_files:
        return None
    slices = [pydicom.dcmread(f) for f in dcm_files]
    slices.sort(key=lambda s: float(getattr(s, 'InstanceNumber', 0)))
    volume = np.stack([s.pixel_array for s in slices], axis=-1).astype(np.float32)
    slopes     = np.array([float(getattr(s, 'RescaleSlope',     1.0)) for s in slices])
    intercepts = np.array([float(getattr(s, 'RescaleIntercept', 0.0)) for s in slices])
    volume = volume * slopes[np.newaxis, np.newaxis, :] + intercepts[np.newaxis, np.newaxis, :]
    volume = np.clip(volume, -150, 250)
    volume = (volume + 150) / 400.0  # [0, 1]
    factors = tuple(128 / s for s in volume.shape)
    volume = zoom(volume, factors, order=1).astype(np.float16)
    return volume

df_all = pd.read_csv(TRAIN_CSV)
patient_ids = df_all['patient_id'].unique()
missing = []

for pid in tqdm(patient_ids, desc='DICOM -> NPY'):
    out_path = os.path.join(CACHE_DIR, f'{pid}.npy')
    if os.path.exists(out_path):
        continue
    vol = dicom_to_npy(pid)
    if vol is not None:
        np.save(out_path, vol)
    else:
        missing.append(pid)

print(f'Tamamlandi. Eksik: {len(missing)} hasta')


In [ ]:
from model_architecture import FastRSNADataset, RSNAResNetCBAMClassifier


df_train = pd.read_csv(TRAIN_CSV)

gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(df_train, groups=df_train['patient_id']))
train_idx, val_idx = splits[0]
df_tr, df_va = df_train.iloc[train_idx], df_train.iloc[val_idx]

# FastRSNADataset: .npy okur, DICOM degil
train_ds = FastRSNADataset(df_tr, CACHE_DIR, augment=True)
val_ds   = FastRSNADataset(df_va, CACHE_DIR, augment=False)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)

# Model
model = RSNAResNetCBAMClassifier(num_classes=13)
if torch.cuda.device_count() > 1:
    print(f'DataParallel: {torch.cuda.device_count()} GPU aktif')
    model = nn.DataParallel(model)
model = model.to(device)

# Loss, Optimizer, Scheduler
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
num_epochs = 50
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=2e-4,
    steps_per_epoch=len(train_loader),
    epochs=num_epochs, pct_start=0.1
)
scaler = GradScaler(device=device.type)

# Egitim Dongusu
best_val_loss = float('inf')
print('\n--- RSNA 2023 Egitimi Basliyor ---')

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} Train'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * images.size(0)
    train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} Val'):
            images, labels = images.to(device), labels.to(device)
            with autocast(device_type=device.type):
                outputs = model(images)
                loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
    val_loss = running_loss / len(val_loader.dataset)

    lr_now = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch+1:02d}/{num_epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {lr_now:.2e}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state, '/kaggle/working/rsna2023_best_attention_model.pth')
        print('  => Best model kaydedildi')


In [ ]:
# Cihaz Kontrolü
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Aktif Cihaz: {device}")

# T4 x2: kaç GPU var?
n_gpu = torch.cuda.device_count()
print(f"GPU sayısı: {n_gpu}")

# Kaggle Dizin Yolları
DATA_DIR = "/kaggle/input/competitions/rsna-2023-abdominal-trauma-detection"
TRAIN_IMAGES_DIR = os.path.join(DATA_DIR, "train_images")
TRAIN_CSV = os.path.join(DATA_DIR, "train_2024.csv")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)

print(f"Toplam Hasta Sayısı (Train): {df_train['patient_id'].nunique()}")
print(f"Sütun Sayısı: {len(df_train.columns)}")
df_train.head()

**1. Sınıf Dengesizliği (Class Imbalance)**# 

In [ ]:
# Organ bazlı hasar (injury) oranlarını hesaplayalım
injury_cols = ['bowel_injury', 'extravasation_injury', 'kidney_low', 'kidney_high', 
               'liver_low', 'liver_high', 'spleen_low', 'spleen_high']

injury_counts = df_train[injury_cols].sum()

plt.figure(figsize=(12, 6))
sns.barplot(x=injury_counts.values, y=injury_counts.index, palette="Reds_r")
plt.title("Veri Setindeki Yaralanma/Hasar Sınıflarının Dağılımı")
plt.xlabel("Hasta Sayısı")
plt.show()

Keşif: Grafiği çizdiğinizde göreceksiniz ki kidney_high veya bowel_injury vakaları toplam veri setinde oldukça az yer kaplar. Modelin bu sınıfları "görmezden gelmesini" engellemek için daha önce bahsettiğimiz ağırlıklı loss (pos_weight) parametresi hayati önem taşıyacaktır.

Ağırlıklı Önem Oranı: Yarışmanın resmi metriğinde bowel_injury ve extravasation_injury yanlış tahmin edilirse skorunuzu çok kötü etkiler (ceza puanı yüksektir). Analizde bu sınıfların az olduğunu gördüğümüz için, loss fonksiyonuna kesinlikle pozitif ağırlık eklemeliyiz.

2. Çoklu Organ Yaralanmaları (Multi-Label Korelasyonu)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_train[injury_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Hasar Türleri Arasındaki İlişki (Korelasyon) Matrisi")
plt.show()

Keşif: Eğer iki hasar türü arasında pozitif bir korelasyon varsa, modelimiz bu organların özelliklerini ortak katmanlarda öğrenirken birbirini destekleyecek şekilde eğtilebilir.

3. Görüntü (Slice) Sayılarındaki Tutarsızlık

In [ ]:
df_meta = pd.read_csv(os.path.join(DATA_DIR, "train_series_meta.csv"))

# Her serinin kaçar görüntü içerdiğini anlamak için (Örnek olarak ilk birkaç seriyi analiz edebiliriz)
print(df_meta.head())

# Hastaların tarama sayıları (Her hastanın birden fazla serisi var mı?)
series_per_patient = df_meta.groupby('patient_id')['series_id'].count()
print(f"\nHasta başına düşen ortalama tarama (seri) sayısı: {series_per_patient.mean():.2f}")

In [ ]:
# --- FAZ 1 & FAZ 2: ÖN İŞLEME VE VERİ ARTIRIMI (AUGMENTATION) ---

# Train Transform: Ön işleme + Faz 2 Regülasyonları
train_transforms = Compose([
    EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
    # Faz 1: HU Clipping [-150, 250] ve [0, 1] normalizasyonu
    ScaleIntensityRanged(keys=["image"], a_min=-150, a_max=250, b_min=0.0, b_max=1.0, clip=True),
    # Faz 1: 128x128x128 Hacimsel Yeniden Boyutlandırma
    Resized(keys=["image"], spatial_size=(128, 128, 128)),
    
    # Faz 2: Farklı tarama oryantasyonları ve gürültüler için Augmentation adımları
    RandRotate90d(keys=["image"], prob=0.5, spatial_axes=(0, 1)),
    RandFlipd(keys=["image"], prob=0.5, spatial_axis=0),
    RandZoomd(keys=["image"], prob=0.3, min_zoom=0.9, max_zoom=1.1, mode="trilinear"),
    RandGaussianNoised(keys=["image"], prob=0.2, mean=0.0, std=0.05),
    
    ToTensord(keys=["image"])
])

# Validation Transform: Sadece Ön İşleme (Augmentation uygulanmaz)
val_transforms = Compose([
    EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
    ScaleIntensityRanged(keys=["image"], a_min=-150, a_max=250, b_min=0.0, b_max=1.0, clip=True),
    Resized(keys=["image"], spatial_size=(128, 128, 128)),
    ToTensord(keys=["image"])
])

Dinamik Derinlik: Slice sayılarındaki aşırı dalgalanma yüzünden Faz 1'de uyguladığımız Resized(spatial_size=(128, 128, 128)) adımı hayat kurtarıcıdır. Az slice'lı hastaları doldurur, çok slice'lı hastaları sıkıştırarak modeli standart bir boyuta zorlar.


In [ ]:

# Modeli tekrar ana döngüye bağlayalım
from model_architecture import RSNAResNetCBAMClassifier


model = RSNAResNetCBAMClassifier(num_classes=13).to(device)
print("Süper! İç değişken bağımlılıkları tamamen ezildi ve model eğitime hazır hale getirildi. 🚀")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(loader, desc="Eğitim Adımı"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # Mixed Precision (FP16) ile GPU Hızlandırma
        with autocast(device_type=device.type):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        # 1. Loss değerini ölçekle ve geriye yayılımı yap (Doğru)
        scaler.scale(loss).backward()
        
        # 2. Hatalı olan scaler.scale(optimizer).step() yerine DOĞRU kullanım:
        scaler.step(optimizer)
        
        # 3. Scaler faktörünü bir sonraki adım için güncelle
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Doğrulama Adımı"):
            images, labels = images.to(device), labels.to(device)
            
            # Hatalı olan boş autocast() yerine cihaz tipini belirtiyoruz:
            with autocast(device_type=device.type):
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

In [ ]:
from model_architecture import RSNA20233DDataset

# ── Veri ──────────────────────────────────────────────────────────────────
df_train = pd.read_csv(TRAIN_CSV)

gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(df_train, groups=df_train["patient_id"]))
train_idx, val_idx = splits[0]
df_tr, df_va = df_train.iloc[train_idx], df_train.iloc[val_idx]

train_ds = RSNA20233DDataset(df_tr, TRAIN_IMAGES_DIR, transform=train_transforms)
val_ds   = RSNA20233DDataset(df_va, TRAIN_IMAGES_DIR, transform=val_transforms)

# num_workers=4 (Kaggle 4 core), persistent_workers epoch arası restart önler
# prefetch_factor=2 GPU boşta kalmasın diye sonraki batch'i önceden hazırlar
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)

# ── Model — DataParallel ile T4 x2 ───────────────────────────────────────
model = RSNAResNetCBAMClassifier(num_classes=13)
if torch.cuda.device_count() > 1:
    print(f"DataParallel: {torch.cuda.device_count()} GPU aktif")
    model = nn.DataParallel(model)
model = model.to(device)

# ── Loss, Optimizer, Scheduler ───────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
num_epochs = 50
# OneCycleLR: warmup + cosine decay, sabit LR'den ~2× hızlı yakınsama
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=2e-4,
    steps_per_epoch=len(train_loader),
    epochs=num_epochs, pct_start=0.1
)
scaler = GradScaler(device=device.type)

# ── Eğitim Döngüsü ────────────────────────────────────────────────────────
best_val_loss = float("inf")
print("--- RSNA 2023 Eğitimi Başlıyor ---")

for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} Train"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * images.size(0)
    train_loss = running_loss / len(train_loader.dataset)

    # --- Validation ---
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} Val"):
            images, labels = images.to(device), labels.to(device)
            with autocast(device_type=device.type):
                outputs = model(images)
                loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
    val_loss = running_loss / len(val_loader.dataset)

    lr_now = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1:02d}/{num_epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {lr_now:.2e}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # DataParallel sarmalıysa .module ile gerçek model state'i alınır
        state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
        torch.save(state, "/kaggle/working/rsna2023_best_attention_model.pth")
        print("  => Best model kaydedildi")
